In [29]:
%load_ext sql
%reload_ext sql
import prettytable

for name in ["DEFAULT", "PLAIN_COLUMNS", "MSWORD_FRIENDLY", "MARKDOWN", "ORGMODE", "DOUBLE_BORDER", "SINGLE_BORDER", "RST", "RANDOM"]:
    if name not in prettytable.__dict__:
        try:
            prettytable.__dict__[name] = getattr(prettytable.TableStyle, name)
        except AttributeError:
            pass

%config SqlMagic.style = 'DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [30]:
%sql sqlite:///celebral.db

In [9]:
%pip install ipython-sql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%%sql
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers ( 
    customer_id   INT           PRIMARY KEY, 
    first_name    VARCHAR(50)   NOT NULL, 
    last_name     VARCHAR(50)   NOT NULL, 
    email         VARCHAR(100)  UNIQUE NOT NULL, 
    city          VARCHAR(50)   NOT NULL, 
    state         VARCHAR(50)   NOT NULL, 
    join_date     DATE          NOT NULL, 
    is_premium    BOOLEAN       DEFAULT FALSE 
); 
 
-- Index for filtering by city/state 
CREATE INDEX idx_customers_city ON customers(city); 
CREATE INDEX idx_customers_state ON customers(state);

 * sqlite:///celebral.db
Done.
Done.
Done.
Done.
Done.
Done.
Done.


[]

In [12]:
%%sql
CREATE TABLE products ( 
    product_id    INT           PRIMARY KEY, 
    product_name  VARCHAR(100)  NOT NULL, 
    category      VARCHAR(50)   NOT NULL, 
    brand         VARCHAR(50)   NOT NULL, 
    unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
    stock_qty     INT           NOT NULL  DEFAULT 0  CHECK (stock_qty >= 0) 
); 
 
-- Index for filtering by category 
CREATE INDEX idx_products_category ON products(category); 

 * sqlite:///celebral.db
Done.
Done.


[]

In [13]:
%%sql
CREATE TABLE orders ( 
    order_id      INT           PRIMARY KEY, 
    customer_id   INT           NOT NULL, 
    order_date    DATE          NOT NULL, 
    status        VARCHAR(20)   NOT NULL  DEFAULT 'Pending' 
                  CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')), 
    total_amount  DECIMAL(12,2) NOT NULL  CHECK (total_amount >= 0), 
     
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id) 
); 
 
-- Index for date-based filtering and sorting 
CREATE INDEX idx_orders_date ON orders(order_date); 
CREATE INDEX idx_orders_status ON orders(status); 

 * sqlite:///celebral.db
Done.
Done.
Done.


[]

In [14]:
%%sql
CREATE TABLE order_items ( 
    item_id       INT           PRIMARY KEY, 
    order_id      INT           NOT NULL, 
    product_id    INT           NOT NULL, 
    quantity      INT           NOT NULL  CHECK (quantity > 0), 
    unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
    discount_pct  DECIMAL(5,2)  DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100), 
     
    FOREIGN KEY (order_id)   REFERENCES orders(order_id), 
    FOREIGN KEY (product_id) REFERENCES products(product_id) 
); 

 * sqlite:///celebral.db
Done.


[]

In [15]:
%%sql
INSERT INTO customers VALUES 
(101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', TRUE), 
(102, 'Priya',  'Patel',  'priya.p@email.com',  'Ahmedabad', 'Gujarat',     '2024-02-20', FALSE), 
(103, 'Rohan',  'Gupta',  'rohan.g@email.com',  'Delhi',     'Delhi',       '2024-03-10', TRUE), 
(104, 'Sneha',  'Reddy',  'sneha.r@email.com',  'Hyderabad', 'Telangana',   '2024-04-05', FALSE), 
(105, 'Vikram', 'Singh',  'vikram.s@email.com', 'Jaipur',    'Rajasthan',   '2024-05-12', TRUE), 
(106, 'Ananya', 'Iyer',   'ananya.i@email.com', 'Chennai',   'Tamil Nadu',  '2024-06-18', FALSE), 
(107, 'Karan',  'Mehta',  'karan.m@email.com',  'Pune',      'Maharashtra', '2024-07-22', TRUE), 
(108, 'Divya',  'Nair',   'divya.n@email.com',  'Kochi',     'Kerala',      '2024-08-30', FALSE); 

 * sqlite:///celebral.db
8 rows affected.


[]

In [16]:
%%sql
INSERT INTO products VALUES 
(201, 'Wireless Earbuds',     'Electronics', 'BoAt',          1499.00, 250), 
(202, 'Cotton T-Shirt',       'Clothing',    'Levis',         799.00,  500), 
(203, 'Smart Watch',          'Electronics', 'Noise',         2999.00, 150), 
(204, 'Running Shoes',        'Clothing',    'Nike',          4599.00, 120), 
(205, 'Bluetooth Speaker',    'Electronics', 'JBL',           3499.00, 200), 
(206, 'Bedsheet Set',         'Home',        'Spaces',        1299.00, 300), 
(207, 'Laptop Stand',         'Electronics', 'AmazonBasics',  899.00,  180), 
(208, 'Cushion Covers (Set)', 'Home',        'HomeCenter',    599.00,  400); 

 * sqlite:///celebral.db
8 rows affected.


[]

In [17]:
%%sql
INSERT INTO orders VALUES 
(1001, 101, '2024-08-01', 'Delivered',  4498.00), 
(1002, 102, '2024-08-03', 'Delivered',  799.00), 
(1003, 103, '2024-08-05', 'Shipped',    7498.00), 
(1004, 101, '2024-08-10', 'Delivered',  3499.00), 
(1005, 104, '2024-08-12', 'Cancelled',  2999.00), 
(1006, 105, '2024-08-15', 'Delivered',  5898.00), 
(1007, 106, '2024-08-18', 'Pending',    1299.00), 
(1008, 103, '2024-08-20', 'Delivered',  899.00), 
(1009, 107, '2024-08-25', 'Shipped',    6098.00), 
(1010, 108, '2024-08-28', 'Delivered',  1598.00); 

 * sqlite:///celebral.db
10 rows affected.


[]

In [18]:
%%sql
INSERT INTO order_items VALUES 
(5001, 1001, 201, 2, 1499.00, 0), 
(5002, 1001, 207, 1, 899.00,  10), 
(5003, 1002, 202, 1, 799.00,  0), 
(5004, 1003, 203, 1, 2999.00, 0), 
(5005, 1003, 204, 1, 4599.00, 5), 
(5006, 1004, 205, 1, 3499.00, 0), 
(5007, 1005, 203, 1, 2999.00, 0), 
(5008, 1006, 201, 1, 1499.00, 10), 
(5009, 1006, 204, 1, 4599.00, 5), 
(5010, 1007, 206, 1, 1299.00, 0), 
(5011, 1008, 207, 1, 899.00,  0), 
(5012, 1009, 205, 1, 3499.00, 0), 
(5013, 1009, 208, 2, 599.00,  15), 
(5014, 1010, 206, 1, 1299.00, 0), 
(5015, 1010, 208, 1, 599.00,  0); 

 * sqlite:///celebral.db
15 rows affected.


[]

## Section - A

In [32]:
%%sql
SELECT * FROM customers;

 * sqlite:///celebral.db
Done.


customer_id,first_name,last_name,email,city,state,join_date,is_premium
101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


In [33]:
%%sql
SELECT first_name, last_name, city from customers;

 * sqlite:///celebral.db
Done.


first_name,last_name,city
Aarav,Sharma,Mumbai
Priya,Patel,Ahmedabad
Rohan,Gupta,Delhi
Sneha,Reddy,Hyderabad
Vikram,Singh,Jaipur
Ananya,Iyer,Chennai
Karan,Mehta,Pune
Divya,Nair,Kochi


In [34]:
%%sql
SELECT DISTINCT(category) FROM products;

 * sqlite:///celebral.db
Done.


category
Clothing
Electronics
Home


Q4. The primary key of each table is the column that uniquely identifies every row in that table:

- customers → customer_id
- products → product_id
- orders → order_id
- order_items → item_id

A primary key must be unique because it is used to distinguish one row from another. If two rows shared the same primary key, the database would not know which row to reference. It must also be NOT NULL because a missing value would make that row impossible to identify reliably.


Q5. In the customers table, the email column has two important constraints:

- UNIQUE, which means no two customers can have the same email address
- NOT NULL, which means the email value cannot be left empty

If a duplicate email is inserted, the database will raise an error and reject the insert. The reason is that the UNIQUE constraint prevents duplicate values in that column.


Q6. A sample INSERT statement is shown below:

%%sql
INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty)
VALUES (209, 'Test Product', 'Electronics', 'TestBrand', -50, 10);

When this is executed, SQLite will reject the row because the unit_price column has a CHECK constraint that requires unit_price > 0. Since -50 is not greater than zero, the insert fails with a constraint violation error.


## Section - B — Filtering & Optimization (WHERE, Indexes)

In [35]:
%%sql
SELECT * FROM orders WHERE status = 'Delivered';    

 * sqlite:///celebral.db
Done.


order_id,customer_id,order_date,status,total_amount
1001,101,2024-08-01,Delivered,4498
1002,102,2024-08-03,Delivered,799
1004,101,2024-08-10,Delivered,3499
1006,105,2024-08-15,Delivered,5898
1008,103,2024-08-20,Delivered,899
1010,108,2024-08-28,Delivered,1598


In [36]:
%%sql
SELECT product_name FROM products
WHERE category = 'Electronics' AND unit_price > 1000;

 * sqlite:///celebral.db
Done.


product_name
Wireless Earbuds
Smart Watch
Bluetooth Speaker


In [31]:
%%sql
SELECT first_name, last_name
FROM customers
WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31' AND state = 'Maharashtra';

 * sqlite:///celebral.db
Done.


first_name,last_name
Aarav,Sharma
Karan,Mehta


In [47]:
%%sql
SELECT order_id
FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-25' AND status != 'Cancelled';

 * sqlite:///celebral.db
Done.


order_id
1001
1002
1003
1004
1006
1007
1008
1009


Q11. The index idx_orders_date helps the database locate rows faster when the query filters by order_date. Instead of scanning the entire orders table, the database can use the index to find matching dates more efficiently.

This improves performance for queries such as:

%%sql
SELECT *
FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-31';


That query would benefit from the index because it is searching by a range of dates.


Q12. The query below would not use the index efficiently because it applies a function to the join_date column:

%%sql
SELECT * FROM customers WHERE YEAR(join_date) = 2024;

Because the database has to calculate YEAR(join_date) for each row, it cannot use the index effectively. A more index-friendly version is:

%%sql
SELECT *
FROM customers
WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31';

This is more SARGable because the condition is written directly against the indexed column without transforming it.


## Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

In [37]:
%%sql
SELECT COUNT(order_id) AS total_orders FROM orders;

 * sqlite:///celebral.db
Done.


total_orders
10


In [38]:
%%sql
SELECT SUM(total_amount) AS total_revenue FROM orders WHERE status = 'Delivered';

 * sqlite:///celebral.db
Done.


total_revenue
17191


In [39]:
%%sql
SELECT AVG(unit_price) AS avg_price FROM products;

 * sqlite:///celebral.db
Done.


avg_price
2024.0


In [40]:
%%sql
SELECT status AS order_status, COUNT(order_id) AS total_orders, SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC;

 * sqlite:///celebral.db
Done.


order_status,total_orders,total_revenue
Delivered,6,17191
Shipped,2,13596
Cancelled,1,2999
Pending,1,1299


In [41]:
%%sql
SELECT MAX(total_amount) AS max_order_value, MIN(total_amount) AS min_order_value FROM orders;

 * sqlite:///celebral.db
Done.


max_order_value,min_order_value
7498,799


In [48]:
%%sql
SELECT AVG(total_amount) AS average_order_value
FROM orders;

 * sqlite:///celebral.db
Done.


average_order_value
3508.5


## Section D — Joins & Relationships 

In [42]:
%%sql
SELECT o.order_id, c.first_name, c.last_name, o.order_date, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;

 * sqlite:///celebral.db
Done.


order_id,first_name,last_name,order_date,total_amount
1001,Aarav,Sharma,2024-08-01,4498
1002,Priya,Patel,2024-08-03,799
1003,Rohan,Gupta,2024-08-05,7498
1004,Aarav,Sharma,2024-08-10,3499
1005,Sneha,Reddy,2024-08-12,2999
1006,Vikram,Singh,2024-08-15,5898
1007,Ananya,Iyer,2024-08-18,1299
1008,Rohan,Gupta,2024-08-20,899
1009,Karan,Mehta,2024-08-25,6098
1010,Divya,Nair,2024-08-28,1598


In [43]:
%%sql
SELECT C.first_name, C.last_name, O.order_id 
FROM customers C
LEFT JOIN orders O ON C.customer_id = O.customer_id;

 * sqlite:///celebral.db
Done.


first_name,last_name,order_id
Aarav,Sharma,1001
Aarav,Sharma,1004
Priya,Patel,1002
Rohan,Gupta,1003
Rohan,Gupta,1008
Sneha,Reddy,1005
Vikram,Singh,1006
Ananya,Iyer,1007
Karan,Mehta,1009
Divya,Nair,1010


In [44]:
%%sql
SELECT O.order_id, P.product_name, OI.quantity, OI.unit_price, OI.discount_pct
FROM orders O
JOIN order_items OI ON O.order_id = OI.order_id
JOIN products P ON OI.product_id = P.product_id;

 * sqlite:///celebral.db
Done.


order_id,product_name,quantity,unit_price,discount_pct
1001,Wireless Earbuds,2,1499,0
1001,Laptop Stand,1,899,10
1002,Cotton T-Shirt,1,799,0
1003,Smart Watch,1,2999,0
1003,Running Shoes,1,4599,5
1004,Bluetooth Speaker,1,3499,0
1005,Smart Watch,1,2999,0
1006,Wireless Earbuds,1,1499,10
1006,Running Shoes,1,4599,5
1007,Bedsheet Set,1,1299,0


Q22. A LEFT JOIN returns all rows from the left table and only the matching rows from the right table. If there is no match, the right-side columns appear as NULL.

For example, using customers as the left table:

```sql
SELECT c.customer_id, c.first_name, o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;
```

A RIGHT JOIN works the opposite way. It returns all rows from the right table and only the matching rows from the left table. A FULL OUTER JOIN combines both left and right results, including unmatched rows from both sides. In practice, it is useful when you want to see all customers and all orders even when some customers have no orders and some orders do not belong to a known customer.


Q23. The foreign key relationships in this schema are:

- orders.customer_id → customers.customer_id
- order_items.order_id → orders.order_id
- order_items.product_id → products.product_id

If you try to insert an order with customer_id = 999, the insert will fail because 999 does not exist in the customers table. This is enforced by the foreign key constraint, which ensures that every order must refer to a valid customer.


## Section E — Advanced Concepts (CASE, ACID, Transactions) 

In [45]:
%%sql
SELECT product_name, unit_price,
    CASE 
        WHEN unit_price < 1000 THEN 'Budget'
        WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
        WHEN unit_price > 3000 THEN 'Premium'
        ELSE 'Unknown'
    END AS price_tier
FROM products;

 * sqlite:///celebral.db
Done.


product_name,unit_price,price_tier
Wireless Earbuds,1499,Mid-Range
Cotton T-Shirt,799,Budget
Smart Watch,2999,Mid-Range
Running Shoes,4599,Premium
Bluetooth Speaker,3499,Premium
Bedsheet Set,1299,Mid-Range
Laptop Stand,899,Budget
Cushion Covers (Set),599,Budget


In [46]:
%%sql
SELECT 
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_count,
    SUM(CASE WHEN status = 'Delivered' THEN 0 ELSE 1 END) AS not_delivered_count
FROM orders;

 * sqlite:///celebral.db
Done.


delivered_count,not_delivered_count
6,4


Q26. ACID stands for:

- A – Atomicity: all parts of a transaction succeed or none of them do
- C – Consistency: the database remains in a valid state before and after the transaction
- I – Isolation: transactions are protected from interference by other transactions
- D – Durability: once a transaction is committed, its changes are permanently saved

A real-world example is a bank transfer. If money is deducted from one account and not added to the other, the system would be inconsistent. ACID properties ensure that the transfer is completed fully, safely, and permanently.


Q27. The complete transaction block is shown below:

%%sql

BEGIN;

INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
VALUES (1011, 102, DATE('now'), 'Pending', 1598.00);

INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
VALUES (5016, 1011, 201, 1, 1499.00, 0);

INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
VALUES (5017, 1011, 208, 1, 599.00, 0);

UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 201;

UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 208;

COMMIT;


If any statement fails, the transaction can be rolled back using ROLLBACK instead of COMMIT.
